# emogist: full LoRA training (Ministral 3 3B)

This notebook trains a small **LoRA adapter** on the full 3,000-example dataset, so Ministral 3 3B learns to turn a sentence into an emoji sentence.

**Before you start:** in Colab, open **Runtime > Change runtime type**, select **T4 GPU**, and save. Then run the cells from top to bottom, one at a time (Shift+Enter).

**If Colab disconnects:** reconnect, select the T4 GPU again, and run every cell from the top. Step 7 continues from the last finished epoch saved in your Google Drive.


## 1. Install the libraries

Colab starts as an empty machine, so we install the training tools. Versions come from Unsloth's official Ministral 3 notebook, and differ slightly from the Qwen ones. This takes a few minutes.

If Colab asks you to restart the session afterwards, accept, then continue from step 2.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==5.3.0
!pip install --no-deps trl==0.22.2

## 2. Settings

Everything you might change between runs is here.

- **Epochs:** how many times the model sees the whole training set.
- **Batch size:** how many examples it looks at before each small weight update.
- **Learning rate:** how big each update is. Too high and training becomes unstable; too low and it barely learns.
- **LoRA rank:** the size of the add-on. Larger can learn more, but overfits more easily.

In [ ]:
import os
import torch

BASE_MODEL = "unsloth/Ministral-3-3B-Instruct-2512"
RUN_NAME = "ministral3b_full"
TRAIN_FILE = "train.jsonl"
DRIVE_DIR = f"/content/drive/MyDrive/emogist/run_{RUN_NAME}"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints"

EPOCHS = 3
BATCH_SIZE = 4
ACCUMULATION_STEPS = 2          # one update every 8 examples, as in the Qwen runs
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 256                # our examples are short
SEED = 3407

# Short on purpose: the behavior should come from the training examples, not a long prompt.
SYSTEM_PROMPT = "Convert the sentence into a short emoji sentence that keeps its gist. Answer with emojis only."

assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Upload the data and connect Google Drive

Run this cell, then choose these two files from your repo:

- `03_dataset/full/train.jsonl`: the sentences and labels to learn from
- `03_dataset/sentences/real_dev.csv`: 100 real sentences the model never trains on, used to score it afterwards

Colab then asks for access to your Google Drive. Checkpoints go to `MyDrive/emogist/`, so a disconnect doesn't lose the run.


In [ ]:
from google.colab import drive, files

needed = [TRAIN_FILE, "real_dev.csv"]
missing = [name for name in needed if not os.path.exists(name)]
if missing:
    print("Please select:", missing)
    files.upload()
missing = [name for name in needed if not os.path.exists(name)]
assert not missing, f"Still missing: {missing}"
print("Data ready:", needed)

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoints go to", DRIVE_DIR)

## 4. Load the base model

This downloads Ministral 3 3B, which is larger than Qwen 2B. Ministral can also read images, but we only train its text layers.

You may see a warning about an "incorrect regex pattern". It's safe to ignore here: with and without that fix, all our training examples and dev prompts produce identical tokens. (The fix's flag crashes with the pinned `transformers` version.)

In [ ]:
from transformers import AutoTokenizer
from unsloth import FastVisionModel

model, _ = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)
text_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Model loaded")

## 5. Prepare the examples

Each example becomes the exact chat format Ministral expects. We turn it into tokens here, because Ministral's template already includes the start token `<s>`, and the default path would add a second one.

As with Qwen, we hold back 10% as a **validation set**.

In [ ]:
import json
from datasets import Dataset

def to_messages(sentence, emojis=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": sentence}]
    if emojis is not None:
        messages.append({"role": "assistant", "content": emojis})
    return messages

def encode(row):
    text = text_tokenizer.apply_chat_template(to_messages(row["sentence"], row["emojis"]), tokenize=False)
    ids = text_tokenizer(text, add_special_tokens=False)["input_ids"][:MAX_LENGTH]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": list(ids)}

with open(TRAIN_FILE, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

split = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)
columns = split["train"].column_names
train_data = split["train"].map(encode, remove_columns=columns)
validation_data = split["test"].map(encode, remove_columns=columns)

sample = train_data[0]["input_ids"]
assert sample[0] == text_tokenizer.bos_token_id and sample[1] != text_tokenizer.bos_token_id, "Start token problem"
assert sample[-1] == text_tokenizer.eos_token_id, "End token missing"
print(f"{len(train_data)} training examples, {len(validation_data)} validation examples\n")
print(text_tokenizer.decode(sample))

## 6. Attach the LoRA adapter

The roughly 3.9 billion original weights are now frozen. Only the small adapter will train. The printed line shows how few weights that is.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # text only
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 7. Train

The table shows the **training loss** regularly and the **validation loss** after each epoch. Lower is better.

- Both fall: it's learning.
- Training falls while validation rises: it's overfitting.
- Neither moves, or a value becomes `nan`: something is wrong, so stop and send me the output.

Each epoch is saved to Google Drive. At the end, the notebook keeps the epoch with the **lowest validation loss**, which may not be the last one. Run this step once per session.


In [ ]:
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported

# Rerunning this cell would keep training the same model and spoil the comparison.
assert "trainer" not in globals(), "Already trained: use Runtime > Restart session, then run from step 2"

FastVisionModel.for_training(model)
import hashlib
from transformers.trainer_utils import get_last_checkpoint

# A resume must continue this exact run: same data, same settings.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
run_config = {"run": RUN_NAME, "base_model": BASE_MODEL, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
              "accumulation_steps": ACCUMULATION_STEPS, "learning_rate": LEARNING_RATE, "lora_rank": LORA_RANK,
              "max_length": MAX_LENGTH, "seed": SEED, "system_prompt": SYSTEM_PROMPT,
              "train_sha256": hashlib.sha256(open(TRAIN_FILE, "rb").read()).hexdigest()}
config_file = f"{CHECKPOINT_DIR}/run_config.json"
if os.path.exists(config_file):
    with open(config_file) as f:
        assert json.load(f) == run_config, f"{CHECKPOINT_DIR} belongs to a different run: rename it or change RUN_NAME"
else:
    with open(config_file, "w") as f:
        json.dump(run_config, f, indent=1)
resume_from = get_last_checkpoint(CHECKPOINT_DIR)
print("Resuming from", resume_from) if resume_from else print("Starting from epoch 0")

new_trainer = SFTTrainer(
    model=model,
    tokenizer=text_tokenizer,
    data_collator=DataCollatorForSeq2Seq(text_tokenizer, padding=True, label_pad_token_id=-100),
    train_dataset=train_data,
    eval_dataset=validation_data,
    args=SFTConfig(
        max_length=MAX_LENGTH,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=10,
        lr_scheduler_type="linear",
        weight_decay=0.001,
        optim="adamw_8bit",
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=EPOCHS,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        output_dir=CHECKPOINT_DIR,
        report_to="none",
        # Our examples are already tokenized in step 5.
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)
result = new_trainer.train(resume_from_checkpoint=resume_from)
trainer = new_trainer

# One validation per epoch, no more: this catches a run that was trained twice.
evals = [e for e in trainer.state.log_history if "eval_loss" in e]
assert len(evals) == EPOCHS, f"Expected {EPOCHS} validations, found {len(evals)}: stop and send me the output"
best = min(evals, key=lambda e: e["eval_loss"])
for e in evals:
    print(f"Epoch {e['epoch']:.0f}: validation loss {e['eval_loss']:.4f}" + ("  <- kept" if e is best else ""))
print(f"This session trained for {result.metrics['train_runtime'] / 60:.1f} minutes")

## 8. Try it

A quick look before the real scoring. Feel free to change these sentences.

In [ ]:
assert "trainer" in globals(), "Step 7 has not finished in this session"
FastVisionModel.for_inference(model)

def to_emojis(sentence):
    prompt = text_tokenizer.apply_chat_template(to_messages(sentence), tokenize=False,
                                                add_generation_prompt=True)
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False, use_cache=True,
                                eos_token_id=text_tokenizer.eos_token_id,
                                pad_token_id=text_tokenizer.pad_token_id)
    answer = output[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(answer, skip_special_tokens=True).strip()

for sentence in ["I lost my wallet on the bus this morning.",
                 "Ma sœur m'a dit qu'elle arrivait demain.",
                 "It took me twenty minutes to find my car."]:
    print(sentence, "->", to_emojis(sentence))

## 9. Run it on the dev set

The model labels the 100 dev sentences it has never trained on. The file goes back to your Mac, where we score it with the same round-trip scorer as every other model.

In [ ]:
import csv

with open("real_dev.csv", newline="", encoding="utf-8") as f:
    dev = list(csv.DictReader(f))

for i, row in enumerate(dev, 1):
    row["output"] = to_emojis(row["sentence"])
    if i % 25 == 0:
        print(f"{i}/{len(dev)}")

outputs_file = f"dev_outputs_{RUN_NAME}.csv"
with open(outputs_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "lang", "sentence", "output"], extrasaction="ignore")
    writer.writeheader()
    writer.writerows(dev)
print("Saved", outputs_file)

## 10. Save your results

**Download this zip.** A copy also goes to your Google Drive. It contains the best adapter, the dev outputs, the loss history and the settings.

Put it in `04_fine_tuning/runs/` in your repo and tell me when it's there. Then disconnect the runtime.


In [ ]:
import shutil

run_dir = f"run_{RUN_NAME}"
os.makedirs(run_dir, exist_ok=True)
model.save_pretrained(f"{run_dir}/adapter")
text_tokenizer.save_pretrained(f"{run_dir}/adapter")
shutil.copy(outputs_file, run_dir)
with open(f"{run_dir}/loss_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=1)
with open(f"{run_dir}/settings.json", "w") as f:
    json.dump({"train_set": "full", "best_epoch": best["epoch"], "best_eval_loss": best["eval_loss"], "base_model": BASE_MODEL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS,
               "learning_rate": LEARNING_RATE, "lora_rank": LORA_RANK, "max_length": MAX_LENGTH,
               "seed": SEED, "system_prompt": SYSTEM_PROMPT}, f, indent=1)

archive = shutil.make_archive(run_dir, "zip", run_dir)
shutil.copy(archive, DRIVE_DIR)
print("Downloading", archive)
files.download(archive)